# Scaling and Transformation

This notebook is a compact but deep reference for data scaling and transformation in machine learning and statistics.

What you get:
- Core theory and intuition
- Exact formulas
- When to use each method
- Assumptions and failure cases
- Model-specific recommendations
- Leakage-safe workflow and best practices

## 1) Why scaling and transformation matter

Real-world features often have different units, ranges, and distributions.

Examples:
- Income in dollars (0 to 1,000,000+)
- Age in years (0 to 100)
- Ratios in [0, 1]

If you train distance-based or gradient-based models directly such as Linear Regression , Logistic Regression KNN etc, large-scale features can dominate optimization and distance computations.

Scaling and transformations help by:
- Improving numerical stability
- Speeding gradient convergence
- Making coefficients more comparable
- Reducing skewness and heavy tails
- Making patterns more linear and variance more stable

## 2) Core definitions

- Scaling: changes magnitude/range of variables (usually monotonic linear mapping).
- Transformation: changes distributional shape or relationship structure (can be nonlinear).
- Normalization can mean either scaling to range or normalizing vectors to unit norm, so always clarify context.

Let feature values be $x_1, x_2, \ldots, x_n$.

Sample statistics:
- Mean: $\mu = \frac{1}{n}\sum_{i=1}^{n}x_i$
- Standard deviation: $s = \sqrt{\frac{1}{n-1}\sum_{i=1}^{n}(x_i-\mu)^2}$
- Median: middle value after sorting
- IQR(Inter Quantile Range): $Q_3 - Q_1$ (75th percentile - 25th percentile)

## 3) Linear scaling methods

### 3.1 Standardization (Z-score scaling)
Formula:
$z = \frac{x - \mu}{\sigma}$

applied:
- When we want mean ($\mu $) of 0 and standard deviation (s) of 1

Properties:
- Mean approximately 0
- Standard deviation approximately 1
- Keeps distribution shape (just shifts and rescales)

Best for:
- Logistic regression
- Linear regression (especially with regularization)
- SVM
- Neural networks
- PCA

Caution:
- Sensitive to outliers because $\mu$ and $\sigma$ are outlier-sensitive.

### 3.2 Min-Max scaling
Formula:
$x_{\text{scaled}} = \frac{x - x_{\min}}{x_{\max} - x_{\min}}$

General range $[a,b]$:
$x_{\text{scaled}} = a + \frac{(x - x_{\min})(b-a)}{x_{\max} - x_{\min}}$

applied:
- when algorithms require bounded inputs usually [0,1], such as Neural Networks, K-Nearest Neighbors (KNN), and distance-based models

Properties:
- Bounded range (commonly $[0,1]$)
- Preserves ordering

Best for:
- Neural nets when bounded inputs help
- Distance methods where range comparability is important

Caution:
- Very sensitive to extreme min and max values.

### 3.3 Robust scaling
Formula:
$x_{\text{robust}} = \frac{x - \text{median}}{\text{IQR}}$

applied:
- when datasets contain significant outliers or extreme values which can not be removed as they are essential but they would disproportionately influence methods like standard scaling (mean/std) or min-max scaling so we use Robust scaling.It uses the median and the interquartile range (IQR), making it resilient to outliers while maintaining the structure of the data distribution

Properties:
- Uses robust statistics
- Less affected by outliers

Best for:
- Heavy-tailed data
- Data with strong outliers

### 3.4 MaxAbs scaling
Formula:
$x_{\text{scaled}} = \frac{x}{\max(|x|)}$

applied:
- Max Absolute Scaling is used to scale features to a range of [-1,1].It is best applied to sparse data (many zeros) or when data is already centered around zero, as it preserves sparsity and avoids shifting the mean

Properties:
- Output in $[-1, 1]$
- Preserves sparsity (important for sparse matrices)

Best for:
- Sparse high-dimensional features (for example text vectors)

## 4) Vector normalization (sample-wise)

For a sample vector $\mathbf{v}=[v_1,v_2,\ldots,v_d]$:

- $L_2$ normalization: $\mathbf{v}_{\text{norm}} = \frac{\mathbf{v}}{\|\mathbf{v}\|_2}$, where $\|\mathbf{v}\|_2 = \sqrt{\sum_{j=1}^{d} v_j^2}$
- $L_1$ normalization: $\mathbf{v}_{\text{norm}} = \frac{\mathbf{v}}{\|\mathbf{v}\|_1}$, where $\|\mathbf{v}\|_1 = \sum_{j=1}^{d} |v_j|$

Use when direction matters more than magnitude, for example cosine similarity pipelines.

Important: this is different from feature-wise scaling.

# 📊 Skewness (Statistical Measure)

## 🔍 Definition
**Skewness** is a statistical measure of the **asymmetry** of a probability distribution around its mean.

- Indicates whether data is **left-skewed (negative)** or **right-skewed (positive)**
- A perfectly symmetrical distribution has **zero skewness**

---

## 📐 Skewness Formulas

### 1) Skewness in terms of moments ($\gamma_1$)
$$
\gamma_1 = \frac{\mu_3}{\mu_2^{3/2}}
$$

Since $\mu_2 = \sigma^2$, this is also:
$$
\gamma_1 = \frac{E[(X-\mu)^3]}{\sigma^3}
$$

### 2) Pearson moment coefficient relation
$$
\beta_1 = \frac{\mu_3^2}{\mu_2^3}
$$

$$
\gamma_1 = \pm\sqrt{\beta_1}
$$

Sign rule:
- Use **+** for right-skewed distributions.
- Use **-** for left-skewed distributions.

---

## 🧾 Meaning of every term

- $X$: random variable
- $\mu = E[X]$: mean of the distribution
- $E[\cdot]$: expectation operator (population average)
- $\mu_r = E[(X-\mu)^r]$: $r$-th central moment
- $\mu_2$: second central moment (variance), so $\mu_2 = \sigma^2$
- $\mu_3$: third central moment
- $\sigma$: standard deviation, $\sigma = \sqrt{\mu_2}$
- $\beta_1$: Pearson's first skewness moment ratio (non-negative)
- $\gamma_1$: skewness coefficient with sign (direction of skewness)

---

## 🧠 Why Skewness Matters

- Helps understand **data distribution shape**
- Detects **presence of outliers**
- Guides **preprocessing decisions** (e.g., log transform)
- Influences **model selection**

---

## ⚖️ Types of Skewness

### 🔹 Positive Skew (Right-Skewed)
- Tail is longer on the **right side**
- If the value of outliers are very high it causes mean to increase
- Relationship:
$$
\text{Mean} > \text{Median} > \text{Mode}
$$

---

### 🔹 Negative Skew (Left-Skewed)
- Tail is longer on the **left side**
- If the value of outliers are very low it causes mean to decrease
- Relationship:
$$
\text{Mean} < \text{Median} < \text{Mode}
$$

---

### 🔹 Zero Skew (Symmetric Distribution)
- Balanced on both sides
- Example: Normal distribution
- Relationship:
$$
\text{Mean} = \text{Median} = \text{Mode}
$$

---

## 📌 Key Takeaways

- **Positive skew -> right tail longer**
- **Negative skew -> left tail longer**
- **Zero skew -> symmetric data**
- Strong skew often requires **transformation (log / power transform)**

---

## 🚀 Practical Insight

- Right-skewed data -> try **log transform**
- Extreme skew -> consider **Yeo-Johnson / Box-Cox**
- Always **visualize (histogram / KDE)** before deciding

## 5) Distribution-shaping transformations

These are often nonlinear and aim to reduce skewness, stabilize variance, or linearize relationships.

### 5.1 Log transform
Common forms:
- $y=\log(x)$ for $x>0$
- $y=\log(1+x)$ for $x\ge 0$ (this is called log1p transform)

Effects:
- Compresses large values
- Reduces right skew
- Converts multiplicative effects to additive effects

Interpretation in linear models:
- If target is log-transformed, coefficient $\beta$ for feature $x$ roughly means a 1-unit increase in $x$ changes target by about $100\beta\%$ (small-$\beta$ approximation).

Caution:
- Undefined for non-positive $x$ (unless shifted or using Yeo-Johnson).

### 5.2 Log1p transform
Formula:
- $y = \log(1+x)$, for $x\ge 0$

Why use it:
- Handles zeros safely ($\log(0)$ is undefined but $\log1p(0)=0$)
- Numerically stable for very small $x$
- Common for sparse counts, clicks, and monetary features with many zeros

### 5.3 Square-root transform
$y=\sqrt{x}$, $x\ge 0$

- Mild skew reduction
- Useful for count-like data

### 5.4 Reciprocal transform
$y=\frac{1}{x}$, $x\ne 0$

- Strongly compresses large values
- Can invert ordering direction (higher $x$ becomes lower $y$)

### 5.5 Power transform family
General: $y=x^{\lambda}$ (with conventions near $\lambda=0$)

#### Box-Cox ($x>0$ only)
$y(\lambda)=\frac{x^{\lambda}-1}{\lambda}$, if $\lambda\ne 0$
$y(0)=\log(x)$

- Lambda selected to make data closer to normal and variance more stable

#### Yeo-Johnson (works with zero and negative values)
For $x\ge 0$:
$y=\frac{(x+1)^{\lambda}-1}{\lambda}$, if $\lambda\ne 0$
$y=\log(x+1)$, if $\lambda=0$

For $x<0$:
$y=-\frac{((-x+1)^{2-\lambda}-1)}{2-\lambda}$, if $\lambda\ne 2$
$y=-\log(-x+1)$, if $\lambda=2$

- More flexible when features include non-positive values

#### When skewness is high (about $>1$ or $<-1$):
Use:

- log1p for right skew (common in fraud, money, counts)
- Box-Cox for strictly positive data
- Yeo-Johnson works with negatives too (best general choice)

## 6) Rank and quantile-based transformations

### 6.1 Quantile transformation to uniform
Map each value by its empirical CDF $\hat{F}(x)$:
$u=\hat{F}(x)$, so $u$ is approximately $\mathrm{Uniform}(0,1)$.

### 6.2 Quantile transformation to normal
$z=\Phi^{-1}(\hat{F}(x))$
where $\Phi^{-1}$ is the inverse standard normal CDF.

Effects:
- Strongly reduces outlier impact
- Produces controlled distribution shape

Caution:
- Can distort linear relationships and distances between nearby points in original space.

## 7) Which models need scaling the most

Usually scale:
- KNN, KMeans, DBSCAN (distance-based)
- SVM (especially with RBF kernel)
- PCA and other eigendecomposition methods
- Linear/logistic regression with gradient solvers and regularization
- Neural networks

Usually less sensitive:
- Tree models (Decision Tree, Random Forest, Gradient Boosted Trees)

Note:
- Even tree models can benefit from transformations if extreme skew or outliers affect split usefulness.

## 8) Target variable transformation

For regression, transforming $y$ can improve error structure and fit.

Example:
- Train on $y_t = \log1p(y)$
- Predict $\hat{y}_t$
- Inverse transform: $\hat{y} = \mathrm{expm1}(\hat{y}_t)$

Benefits:
- Handles right-skewed targets
- Reduces heteroscedasticity

Evaluate on original scale when business interpretation requires it.

## 9) Data leakage and pipeline-safe fitting

Critical rule:
- Fit scaler/transformer on train split only.
- Apply fitted transformer to validation/test/inference data.

Why:
- Using test statistics during fit leaks information and inflates validation scores.

Pipeline pattern:
1. Split data
2. Fit transformer on X_train
3. Transform X_train and X_valid with same fitted object
4. Train model
5. Evaluate

For mixed types, use a column-wise transformer so only numeric columns are scaled while categorical columns are encoded appropriately.

## 10) Choosing the right method (quick decision guide)

If data is approximately symmetric and low-outlier:
- Standardization

If strong outliers exist:
- Robust scaling

If you need strict bounded range:
- Min-Max scaling

If sparse matrix must stay sparse:
- MaxAbs scaling

If positive and heavily right-skewed:
- Log, Box-Cox, or Yeo-Johnson

If non-positive values and skewed:
- Yeo-Johnson

If you need a specific target distribution (uniform or normal):
- Quantile transform

If using cosine similarity/text vectors:
- L2 normalization

## 11) Statistical effects and interpretation notes

- Correlation under linear scaling is unchanged.
- Rank-based transformations preserve order but not original distances.
- Log transform changes additive error structure assumptions.
- Coefficients from transformed features must be interpreted in transformed units.

Approximate percent interpretation:
- If model is $\log(y)=\alpha+\beta x+\varepsilon$, then one unit increase in $x$ changes $y$ by about $100\beta\%$ for small $\beta$.
- Exact multiplicative effect is $\exp(\beta)$.

## 12) Common pitfalls

- Fitting scaler before train-test split (leakage)
- Applying log on zeros/negative values without handling
- Assuming tree models always need scaling (often not necessary)
- Ignoring inverse transform for target predictions
- Using one scaler across fundamentally different feature groups without thought
- Forgetting that transformed distributions can reduce interpretability

## 13) Practical sklearn mapping

Common classes:
- StandardScaler
- MinMaxScaler
- RobustScaler
- MaxAbsScaler
- Normalizer
- PowerTransformer (Box-Cox or Yeo-Johnson)
- QuantileTransformer
- FunctionTransformer (custom log1p, sqrt, etc.)

Fit/transform API pattern:
- fit(X_train)
- transform(X_train), transform(X_test)
- or fit_transform(X_train) followed by transform(X_test)

## 14) Compact formula sheet

1. Standardization: $z=\frac{x-\mu}{s}$
2. Min-Max $[0,1]$: $\frac{x-x_{\min}}{x_{\max}-x_{\min}}$
3. Robust: $\frac{x-\mathrm{median}}{\mathrm{IQR}}$
4. MaxAbs: $\frac{x}{\max(|x|)}$
5. L2 normalize vector $\mathbf{v}$: $\frac{\mathbf{v}}{\|\mathbf{v}\|_2}$
6. Log: $\log(x)$ or $\log(1+x)$
7. Sqrt: $\sqrt{x}$
8. Reciprocal: $\frac{1}{x}$
9. Box-Cox: $\frac{x^{\lambda}-1}{\lambda}$, and $\log(x)$ when $\lambda=0$
10. Quantile to normal: $\Phi^{-1}(\hat{F}(x))$

## 15) Final checklist before model training

- Check skewness and outliers per feature
- Choose scaler/transform per feature type
- Build leakage-safe pipeline
- Validate with cross-validation
- Compare baseline vs transformed performance
- Confirm interpretability and inverse mapping for predictions

This gives a complete conceptual and formula-level foundation for scaling and transformation in ML workflows.

## 16) Pipeline

- Handle missing values
- Reduce skewness (log / power transform)
- Scale (StandardScaler / RobustScaler)
- Apply SMOTE (For imbalanced dataset)
- Train model

## 17) Code snippets: how to use each transform


The following cells show practical usage for each major scaling and transformation method using `numpy`, `pandas`, and `scikit-learn`.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import (
    StandardScaler, MinMaxScaler, RobustScaler, MaxAbsScaler, Normalizer,
    PowerTransformer, QuantileTransformer
)

# Example data (includes positive, zero, and negative values)
df = pd.DataFrame({
    "income": [12000, 18000, 25000, 42000, 90000, 250000],
    "transactions": [0, 1, 2, 3, 7, 25],
    "balance_change": [-500, -120, 0, 50, 300, 900],
    "text_tfidf_like": [0, 0.2, 0, 0.9, 0.1, 0.5],
})

print("Original data:\n", df)

In [ ]:
# 1) Standardization (Z-score)
std_scaler = StandardScaler()
df_std = pd.DataFrame(std_scaler.fit_transform(df), columns=df.columns)
print("\nStandardScaler:\n", df_std.round(3))

# 2) Min-Max scaling to [0,1]
minmax_scaler = MinMaxScaler()
df_minmax = pd.DataFrame(minmax_scaler.fit_transform(df), columns=df.columns)
print("\nMinMaxScaler:\n", df_minmax.round(3))

# 3) Robust scaling (median and IQR)
robust_scaler = RobustScaler()
df_robust = pd.DataFrame(robust_scaler.fit_transform(df), columns=df.columns)
print("\nRobustScaler:\n", df_robust.round(3))

# 4) MaxAbs scaling (great for sparse-like features)
maxabs_scaler = MaxAbsScaler()
df_maxabs = pd.DataFrame(maxabs_scaler.fit_transform(df), columns=df.columns)
print("\nMaxAbsScaler:\n", df_maxabs.round(3))

# 5) Vector normalization (sample-wise L2)
normalizer = Normalizer(norm="l2")
df_l2 = pd.DataFrame(normalizer.fit_transform(df), columns=df.columns)
print("\nL2 Normalizer (row-wise):\n", df_l2.round(3))

In [ ]:
# Use a strictly non-negative column for log/sqrt style transforms
x = df["transactions"].to_numpy(dtype=float)

# 6) Log transform: log(x) requires x > 0, so shift if needed
eps = 1e-6
x_log = np.log(x + eps)
print("log(x + eps):", np.round(x_log, 4))

# 7) Log1p transform: safe for x >= 0 (preferred when zeros exist)
x_log1p = np.log1p(x)
print("log1p(x):", np.round(x_log1p, 4))

# 8) Square-root transform
x_sqrt = np.sqrt(x)
print("sqrt(x):", np.round(x_sqrt, 4))

# 9) Reciprocal transform: avoid division by zero using epsilon
x_recip = 1.0 / (x + eps)
print("1/(x + eps):", np.round(x_recip, 4))

In [ ]:
# 10) Box-Cox and Yeo-Johnson with sklearn PowerTransformer

# Box-Cox needs strictly positive data
income_positive = df[["income"]].copy()
boxcox = PowerTransformer(method="box-cox", standardize=False)
income_boxcox = boxcox.fit_transform(income_positive)
print("\nBox-Cox (income) lambda:", np.round(boxcox.lambdas_[0], 4))
print("Box-Cox transformed income:", np.round(income_boxcox.flatten(), 4))

# Yeo-Johnson supports zero and negative values
yj = PowerTransformer(method="yeo-johnson", standardize=False)
balance_yj = yj.fit_transform(df[["balance_change"]])
print("\nYeo-Johnson (balance_change) lambda:", np.round(yj.lambdas_[0], 4))
print("Yeo-Johnson transformed balance_change:", np.round(balance_yj.flatten(), 4))

# 11) Quantile transforms
qt_uniform = QuantileTransformer(output_distribution="uniform", random_state=42, n_quantiles=min(len(df), 1000))
qt_normal = QuantileTransformer(output_distribution="normal", random_state=42, n_quantiles=min(len(df), 1000))

income_uniform = qt_uniform.fit_transform(df[["income"]])
income_normal = qt_normal.fit_transform(df[["income"]])

print("\nQuantile -> Uniform (income):", np.round(income_uniform.flatten(), 4))
print("Quantile -> Normal  (income):", np.round(income_normal.flatten(), 4))